# 04 — Spatial Disparity Index

**SDI Tehran Pipeline · Notebook 4 of 5**

## Purpose
Compute the Spatial Disparity Index (SDI) for each periphery segment. The SDI is a weighted composite of normalised attribute scores. Weights are derived from model performance (Cramér's V or R²), normalised to sum to 1. High SDI indicates better life quality.

## Weighting scheme
SDI = Σ (normalised_score_i × weight_i)

where weight_i = performance_score_i / Σ performance_scores.

**Rationale:** Attributes predicted with higher model confidence receive proportionally more influence on the index. This ties SDI credibility to empirical fit.

## Inputs
- `predictions_joined.csv`
- `model_scores.json`
- `category_keys.json`

## Outputs
- `disparity_index.csv` — segment ID, per-attribute 0–1 scores, raw SDI, normalised SDI

## 0 · Config & imports

In [1]:
import sys, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, '..')
from config import (
    PREDICTIONS_JOINED_CSV, MODEL_SCORES_JSON, CATEGORY_KEYS_JSON,
    DISPARITY_INDEX_CSV, CATEGORICAL_TARGETS, CONTINUOUS_TARGETS, TARGET_ATTRS,
    SEG_ID_COL
)
print("Config loaded.")

Config loaded.


## 1 · Load data

In [2]:
df = pd.read_csv(PREDICTIONS_JOINED_CSV)

with open(MODEL_SCORES_JSON) as f:
    model_scores = json.load(f)

with open(CATEGORY_KEYS_JSON) as f:
    category_keys = json.load(f)
# Drop the comment key if present
category_keys = {k: v for k, v in category_keys.items() if not k.startswith("_")}

# SEG_ID_COL = "segment_id"
print(f"Loaded {len(df):,} periphery segments")
print(f"Model scores: {json.dumps(model_scores, indent=2)}")

Loaded 279,793 periphery segments
Model scores: {
  "policy_era": {
    "metric": "cramers_v",
    "score": 0.346
  },
  "dilap_index": {
    "metric": "cramers_v",
    "score": 0.2507
  },
  "landuse": {
    "metric": "cramers_v",
    "score": 0.1058
  },
  "mean_area": {
    "metric": "r2",
    "score": -0.00013765413987143482
  },
  "perimeter": {
    "metric": "r2",
    "score": 0.004895614356078659
  },
  "mean_pop": {
    "metric": "r2",
    "score": -4.765688311025862e-05
  }
}


## 2 · Map categorical attributes to numeric scores

**Methodology note:** Each categorical label is mapped to a 0–1 score reflecting scholarly-grounded importance ordering (see `category_keys.json`). This converts qualitative categories to a common numeric scale before weighting. The mappings were defined by the author based on literature review and prior analysis.

In [3]:
for attr in CATEGORICAL_TARGETS:
    keys = category_keys[attr]
    # Cast predicted labels to string for lookup (LabelEncoder may return int or str)
    score_col = f"score_{attr}"
    df[score_col] = df[attr].astype(int).astype(str).map(keys)

    n_unmapped = df[score_col].isnull().sum()
    if n_unmapped > 0:
        print(f"WARNING: {attr} — {n_unmapped} values could not be mapped. Check category_keys.json.")
        print(f"  Unmapped values: {df.loc[df[score_col].isnull(), attr].unique()}")
    else:
        print(f"{attr}: all {len(df):,} values mapped successfully")

policy_era: all 279,793 values mapped successfully
dilap_index: all 279,793 values mapped successfully
  Unmapped values: [7.]


## 3 · Normalise continuous attributes to 0–1

In [4]:
scaler = MinMaxScaler()
cont_scores = scaler.fit_transform(df[CONTINUOUS_TARGETS].values)

for i, attr in enumerate(CONTINUOUS_TARGETS):
    df[f"score_{attr}"] = cont_scores[:, i]
    print(f"score_{attr}: min={df[f'score_{attr}'].min():.3f}  max={df[f'score_{attr}'].max():.3f}")

score_mean_area: min=0.000  max=1.000
score_perimeter: min=0.000  max=1.000
score_mean_pop: min=0.000  max=1.000


## 4 · Compute weights

Weights are drawn directly from `model_scores.json` and normalised to sum to 1. The attribute order follows `TARGET_ATTRS` in `config.py` — this must match the order used when writing `model_scores.json` in notebook 02.

In [5]:
raw_weights = np.array([model_scores[attr]["score"] for attr in TARGET_ATTRS])
weights_normalised = raw_weights / raw_weights.sum()

print("Normalised weights per attribute:")
for attr, w in zip(TARGET_ATTRS, weights_normalised):
    print(f"  {attr:<20}: {w:.4f}  ({w*100:.1f}%)")
print(f"  Sum: {weights_normalised.sum():.6f}")

Normalised weights per attribute:
  policy_era          : 0.4892  (48.9%)
  dilap_index         : 0.3545  (35.4%)
  landuse             : 0.1496  (15.0%)
  mean_area           : -0.0002  (-0.0%)
  perimeter           : 0.0069  (0.7%)
  mean_pop            : -0.0001  (-0.0%)
  Sum: 1.000000


## 5 · Compute SDI

In [6]:
score_cols = [f"score_{attr}" for attr in TARGET_ATTRS]
scores_matrix = df[score_cols].values  # shape: (n_segments, 6)

# Weighted sum
sdi_raw = scores_matrix @ weights_normalised

# Normalise to 0–1
sdi_scaler = MinMaxScaler()
sdi_norm = sdi_scaler.fit_transform(sdi_raw.reshape(-1, 1)).ravel()

df["sdi_raw"]  = sdi_raw
df["sdi"]      = sdi_norm

print(f"SDI computed for {len(df):,} segments")
print(f"  Raw  — min: {sdi_raw.min():.4f}  max: {sdi_raw.max():.4f}  mean: {sdi_raw.mean():.4f}")
print(f"  Norm — min: {sdi_norm.min():.4f}  max: {sdi_norm.max():.4f}  mean: {sdi_norm.mean():.4f}")

SDI computed for 279,793 segments
  Raw  — min: nan  max: nan  mean: nan
  Norm — min: nan  max: nan  mean: nan


## 6 · Save output

In [7]:
output_cols = [SEG_ID_COL] + score_cols + ["sdi_raw", "sdi"]
out = df[output_cols].copy()

out.to_csv(DISPARITY_INDEX_CSV, index=False)
print(f"Saved: {DISPARITY_INDEX_CSV}")
print(f"Shape: {out.shape}")
print(out.describe())

Saved: D:\SID\data\outputs\disparity_index.csv
Shape: (279793, 9)
          Depthmap_R  score_policy_era  score_dilap_index  score_landuse  \
count  279793.000000     279793.000000      279793.000000      279792.00   
mean   240012.330509          0.789459           0.994618           0.25   
std    121735.686486          0.198890           0.068513           0.00   
min         1.000000          0.200000           0.000000           0.25   
25%    136128.000000          0.600000           1.000000           0.25   
50%    252542.000000          0.600000           1.000000           0.25   
75%    343350.000000          1.000000           1.000000           0.25   
max    427099.000000          1.000000           1.000000           0.25   

       score_mean_area  score_perimeter  score_mean_pop        sdi_raw  \
count    279793.000000    279793.000000   279793.000000  279792.000000   
mean          0.476241         0.599431        0.164320       0.780270   
std           0.081764     